# Your First Amphibious Agent

Build your first agent in **5 minutes**. In this tutorial, you'll create the same task using both **Agent mode** (the LLM decides what to do) and **Workflow mode** (you decide what to do), seeing how the same framework supports both paradigms.

## Practical Scenario

We'll build a simple **"weather information assistant"** that looks up weather for cities.

- In **Agent mode**, the LLM autonomously decides which cities to check and how to summarize the results.
- In **Workflow mode**, you define the exact cities and the order in which they are queried.

Same goal, two paradigms, one framework.

## Initialize

First, let's set up the LLM and define a simple weather tool.

In [ ]:
import os
from openai import AsyncOpenAI

api_key = os.environ.get("OPENAI_API_KEY", "your-api-key")
base_url = os.environ.get("OPENAI_BASE_URL", "https://api.openai.com/v1")

In [ ]:
from bridgic.amphibious import AmphibiousAutoma, CognitiveContext, CognitiveWorker, think_unit, step

from bridgic.model import OpenAILlm

llm = OpenAILlm(
    model="gpt-4o-mini",
    api_key=api_key,
    base_url=base_url,
)

## Define a Tool

Let's create a simple mock weather tool that our agent can use.

In [ ]:
from bridgic.core import tool

@tool(description="Get the current weather for a given city")
async def get_weather(city: str) -> str:
    """Mock weather lookup."""
    weather_data = {
        "Tokyo": "Sunny, 22\u00b0C",
        "London": "Cloudy, 15\u00b0C",
        "New York": "Rainy, 18\u00b0C",
        "Paris": "Partly cloudy, 20\u00b0C",
    }
    return weather_data.get(city, f"Weather data not available for {city}")

## Example 1: Agent Mode — Let the LLM Decide

In Agent mode, the LLM autonomously decides which tools to call and in what order. We define a `CognitiveWorker` (the thinking unit) and let it run inside `on_agent`.

In [ ]:
class WeatherAgentMode(AmphibiousAutoma[CognitiveContext]):
    planner = think_unit(
        CognitiveWorker.inline("Look up weather information for relevant cities and provide a summary."),
        max_attempts=5,
    )

    async def on_agent(self, ctx: CognitiveContext):
        await self.planner

Now let's run it. The LLM will decide which cities to look up.

In [ ]:
agent = WeatherAgentMode(llm=llm)
result = await agent.arun(
    goal="Check the weather in Tokyo and London, then give me a brief summary.",
    tools=[get_weather],
)
print(result)

## Example 2: Workflow Mode — You Define the Steps

In Workflow mode, you define the exact execution steps. No LLM reasoning is needed for the flow — each step calls a specific tool with specific arguments.

In [ ]:
class WeatherWorkflowMode(AmphibiousAutoma[CognitiveContext]):
    async def on_agent(self, ctx: CognitiveContext):
        pass  # Required but not used in pure workflow mode

    async def on_workflow(self, ctx: CognitiveContext):
        tokyo_weather = yield step("get_weather", city="Tokyo")
        london_weather = yield step("get_weather", city="London")
        print(f"Tokyo: {tokyo_weather}")
        print(f"London: {london_weather}")

Run the workflow — notice how steps execute in the exact order you defined.

In [ ]:
workflow = WeatherWorkflowMode(llm=llm)
result = await workflow.arun(
    goal="Check weather in Tokyo and London",
    tools=[get_weather],
)
print(result)

## Agent Mode vs. Workflow Mode

| | Agent Mode | Workflow Mode |
|---|---|---|
| **Who decides?** | The LLM | You |
| **Method** | `on_agent` | `on_workflow` |
| **Best for** | Open-ended tasks | Known procedures |
| **Predictability** | Variable | Deterministic |
| **LLM overhead** | Higher (reasoning at each step) | Lower (no flow reasoning) |

- **Agent Mode**: The LLM decides what to do. You define *what to think about*, the LLM handles the rest. Great for open-ended tasks.
- **Workflow Mode**: You define every step. Predictable, repeatable, no LLM reasoning overhead. Great for known procedures.
- Both modes live inside the same `AmphibiousAutoma` class — this is the foundation of the "amphibious" design.

<div style="text-align: center; margin: 2rem 0;">
<hr style="border: none; border-top: 2px solid #e2e8f0;">
</div>

## What have we learnt?

In this tutorial, we built the same weather assistant using two different approaches:

- **AmphibiousAutoma** is the base class for all amphibious agents. It supports both `on_agent` (LLM-driven) and `on_workflow` (deterministic) modes.
- **CognitiveWorker** is the atomic thinking unit. Use `CognitiveWorker.inline("...")` for quick creation, or subclass it for custom behavior.
- **think_unit** is a declarative descriptor that binds a worker with execution parameters (like `max_attempts`). Use `await self.think_unit` inside `on_agent` to trigger a full observe-think-act cycle.
- **step()** is used inside `on_workflow` to define deterministic tool calls. Use `result = yield step("tool_name", **args)` to execute and capture results.
- **arun()** is the main entry point. Pass `goal`, `tools`, and other configuration to start the agent.

Next, we'll dive deeper into the CognitiveWorker — the thinking atom at the heart of every amphibious agent.